# 🩺 Retrain U-Net 3-Kelas pada Mask Folder (Colab GPU)

Melatih ulang U-Net lightweight pada **mask ground-truth Anda** (`data/segmentation/{train,val,test}/masks`)
— bukan regenerasi heuristik, bukan SAM. Tujuan: model yang **benar-benar cocok** dengan mask folder Anda.

- **Self-contained**: arsitektur U-Net didefinisikan inline (tak perlu clone repo → tak ada masalah git).
- **Loss anti-collapse**: weighted Cross-Entropy + multiclass Dice.
- **GPU**: ~15-25 menit (vs 5+ jam di CPU).

> Aktifkan GPU: **Runtime → Change runtime type → T4 GPU**.
> Sebelum mulai: upload folder `data/segmentation/` ke Drive (struktur di sel 2).

## 0. Setup

In [ ]:
import torch, os
print('PyTorch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Aktifkan GPU: Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))
DEV='cuda'


## 1. Data — upload `data/segmentation/` ke Drive

Struktur yang dibutuhkan di Drive (`MyDrive/duck_egg_segmentation/`):
```
duck_egg_segmentation/
├── train/images/*.jpg   train/masks/*.png   (156)
├── val/images/*.jpg     val/masks/*.png     (44)
└── test/images/*.jpg    test/masks/*.png    (24)
```
Mask = PNG nilai 0/1/2 (BG/Vaskular/Embrio). Sel ini menyalin ke runtime.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import shutil, glob
DRIVE_SEG='/content/drive/MyDrive/duck_egg_segmentation'   # ← sesuaikan
assert os.path.exists(DRIVE_SEG), f'Tidak ada: {DRIVE_SEG}'
ROOT='/content/seg'
if os.path.exists(ROOT): shutil.rmtree(ROOT)
shutil.copytree(DRIVE_SEG, ROOT)
for sp in ['train','val','test']:
    ni=len(glob.glob(f'{ROOT}/{sp}/images/*')); nm=len(glob.glob(f'{ROOT}/{sp}/masks/*.png'))
    print(f'{sp}: {ni} images, {nm} masks')


## 2. Arsitektur U-Net Lightweight (inline)

In [ ]:
import torch.nn as nn, torch.nn.functional as F
# Nama atribut SAMA PERSIS dengan src/segmentation/unet_lightweight.py agar state_dict kompatibel
class DoubleConv(nn.Module):
    def __init__(self,i,o):
        super().__init__()
        self.double_conv=nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                                       nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(self,x): return self.double_conv(x)
class Down(nn.Module):
    def __init__(self,i,o): super().__init__(); self.maxpool_conv=nn.Sequential(nn.MaxPool2d(2),DoubleConv(i,o))
    def forward(self,x): return self.maxpool_conv(x)
class Up(nn.Module):
    def __init__(self,i,o,bilinear=True):
        super().__init__(); self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=True); self.conv=DoubleConv(i,o)
    def forward(self,x1,x2):
        x1=self.up(x1); dy=x2.size(2)-x1.size(2); dx=x2.size(3)-x1.size(3)
        x1=F.pad(x1,[dx//2,dx-dx//2,dy//2,dy-dy//2]); return self.conv(torch.cat([x2,x1],1))
class OutConv(nn.Module):
    def __init__(self,i,o): super().__init__(); self.conv=nn.Conv2d(i,o,1)
    def forward(self,x): return self.conv(x)
class UNetLightWeight(nn.Module):
    def __init__(self,n_channels=3,n_classes=3,bilinear=True,dropout_rate=0.2):
        super().__init__()
        self.inc=DoubleConv(n_channels,32); self.down1=Down(32,64); self.down2=Down(64,128); self.down3=Down(128,256)
        self.bottleneck=DoubleConv(256,256)
        self.up1=Up(384,128,bilinear); self.up2=Up(192,64,bilinear); self.up3=Up(96,32,bilinear)
        self.outc=OutConv(32,n_classes); self.dropout=nn.Dropout2d(dropout_rate) if dropout_rate>0 else None
    def forward(self,x):
        x1=self.inc(x); x2=self.down1(x1); x3=self.down2(x2); x4=self.down3(x3)
        x=self.bottleneck(x4)
        if self.dropout: x=self.dropout(x)
        x=self.up1(x,x3); x=self.up2(x,x2); x=self.up3(x,x1); return self.outc(x)
print('UNet siap (state_dict kompatibel dengan repo create_unet_lightweight)')


## 3. Dataset + training (weighted CE + Dice)

In [ ]:
import numpy as np, glob, time
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
SZ, NC = 256, 3
class DS(Dataset):
    def __init__(s,split,aug=False):
        s.aug=aug; s.tt=T.ToTensor(); s.pairs=[]
        for ip in sorted(glob.glob(f'{ROOT}/{split}/images/*')):
            mp=f'{ROOT}/{split}/masks/'+os.path.splitext(os.path.basename(ip))[0]+'.png'
            if os.path.exists(mp): s.pairs.append((ip,mp))
    def __len__(s): return len(s.pairs)
    def __getitem__(s,i):
        ip,mp=s.pairs[i]
        im=np.array(Image.open(ip).convert('RGB').resize((SZ,SZ),Image.BILINEAR))
        mk=np.clip(np.array(Image.open(mp).convert('L').resize((SZ,SZ),Image.NEAREST)),0,NC-1).astype(np.int64)
        if s.aug and np.random.rand()<.5: im=im[:,::-1].copy(); mk=mk[:,::-1].copy()
        if s.aug and np.random.rand()<.3: im=im[::-1].copy(); mk=mk[::-1].copy()
        if s.aug and np.random.rand()<.3:
            k=np.random.randint(1,4); im=np.rot90(im,k).copy(); mk=np.rot90(mk,k).copy()
        return s.tt(Image.fromarray(im)), torch.from_numpy(mk).long()

tr=DS('train',aug=True); va=DS('val')
dl=DataLoader(tr,batch_size=8,shuffle=True,num_workers=2); vdl=DataLoader(va,batch_size=8,num_workers=2)
print('train',len(tr),'val',len(va))

cnt=np.zeros(NC)
for _,y in tr:
    for k in range(NC): cnt[k]+=(y==k).sum().item()
freq=cnt/cnt.sum(); w=np.clip(1.0/(freq+1e-6)/ (1/(freq+1e-6)).sum()*NC, 0.3, 15.0)
cw=torch.tensor(w,dtype=torch.float32,device=DEV); print('freq',freq.round(4),'weights',w.round(2))
ce=nn.CrossEntropyLoss(weight=cw)
def dloss(lo,t,eps=1.):
    p=F.softmax(lo,1); t1=F.one_hot(t,NC).permute(0,3,1,2).float()
    d=(2*(p*t1).sum((0,2,3))+eps)/(p.sum((0,2,3))+t1.sum((0,2,3))+eps); return 1-d.mean()
def vdice(net):
    net.eval(); de=[];dv=[]
    with torch.no_grad():
        for x,y in vdl:
            pr=torch.argmax(net(x.to(DEV)),1).cpu()
            for b in range(x.size(0)):
                g=y[b].numpy(); p=pr[b].numpy()
                dd=lambda c:2*((g==c)&(p==c)).sum()/(((g==c).sum()+(p==c).sum())+1e-9)
                if (g==2).any(): de.append(dd(2))
                if (g==1).any(): dv.append(dd(1))
    return (np.mean(de) if de else 0),(np.mean(dv) if dv else 0)

torch.manual_seed(42); np.random.seed(42)
net=UNetLightWeight().to(DEV)
opt=torch.optim.Adam(net.parameters(),1e-3,weight_decay=1e-4)
sch=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,'max',factor=0.5,patience=8)
EPOCHS=150; best=-1; best_state=None; pat=0; t0=time.time()
for ep in range(1,EPOCHS+1):
    net.train(); tl=0
    for x,y in dl:
        x,y=x.to(DEV),y.to(DEV); opt.zero_grad(); o=net(x); loss=ce(o,y)+dloss(o,y); loss.backward(); opt.step(); tl+=loss.item()*x.size(0)
    de,dv=vdice(net); sc=de+dv; sch.step(sc)
    if sc>best: best=sc; best_state={k:v.cpu().clone() for k,v in net.state_dict().items()}; pat=0
    else: pat+=1
    if ep%5==0 or ep==1: print(f'ep {ep:3d} loss {tl/len(tr):.3f} | val Dice embrio {de:.3f} vask {dv:.3f} | best {best:.3f}')
    if pat>=25: print('early stop'); break
print(f'DONE {time.time()-t0:.0f}s best={best:.3f}')


## 4. Simpan model + evaluasi test + figure perbandingan

In [ ]:
net.load_state_dict(best_state); net.eval()
os.makedirs('/content/out',exist_ok=True)
torch.save({'model_state_dict':best_state,'epoch':EPOCHS,
            'metrics':{'val_best':best},
            'config':{'model':{'lightweight':True,'n_channels':3,'n_classes':3,'bilinear':True,'dropout_rate':0.2}}},
           '/content/out/model.pth')

tt=T.ToTensor()
def predict(ip):
    rgb=np.array(Image.open(ip).convert('RGB').resize((SZ,SZ),Image.BILINEAR))
    with torch.no_grad(): pr=torch.argmax(net(tt(Image.fromarray(rgb)).unsqueeze(0).to(DEV)),1)[0].cpu().numpy()
    return rgb,pr
# Dice test
de=[];dv=[]
for ip in sorted(glob.glob(f'{ROOT}/test/images/*')):
    stem=os.path.splitext(os.path.basename(ip))[0]; mp=f'{ROOT}/test/masks/{stem}.png'
    if not os.path.exists(mp): continue
    _,pr=predict(ip); gt=np.array(Image.open(mp).convert('L').resize((SZ,SZ),Image.NEAREST))
    dd=lambda c:2*((gt==c)&(pr==c)).sum()/(((gt==c).sum()+(pr==c).sum())+1e-9)
    if (gt==2).any(): de.append(dd(2))
    if (gt==1).any(): dv.append(dd(1))
print(f'=== TEST Dice: embrio={np.mean(de):.3f}  vaskular={np.mean(dv):.3f} ===')

import matplotlib.pyplot as plt, matplotlib.patches as mpatches
def col(p):
    o=np.zeros((*p.shape,3),np.uint8); o[p==0]=[35,40,55]; o[p==1]=[230,126,34]; o[p==2]=[46,204,113]; return o
eggs=[('FERTIL',p) for p in sorted(glob.glob(f'{ROOT}/test/images/*'))[:2]]
# pilih juga 1 infertil (mask kosong)
fig,axes=plt.subplots(len(eggs),4,figsize=(13,3.3*len(eggs))); fig.patch.set_facecolor('white')
fig.suptitle(f'U-Net Baru — Citra / GT Mask / Prediksi / Overlay  (Test Dice embrio={np.mean(de):.2f})',
             fontsize=13,fontweight='bold')
for ri,(lab,ip) in enumerate(eggs):
    stem=os.path.splitext(os.path.basename(ip))[0]
    rgb,pr=predict(ip); gt=np.array(Image.open(f'{ROOT}/test/masks/{stem}.png').convert('L').resize((SZ,SZ),Image.NEAREST))
    ov=rgb.copy(); ov[pr==1]=(ov[pr==1]*0.4+np.array([230,126,34])*0.6).astype(np.uint8); ov[pr==2]=(ov[pr==2]*0.4+np.array([46,204,113])*0.6).astype(np.uint8)
    for ci,(im,t) in enumerate([(rgb,'Citra'),(col(gt),'GT Mask'),(col(pr),'Prediksi'),(ov,'Overlay')]):
        ax=axes[ri,ci] if len(eggs)>1 else axes[ci]; ax.imshow(im); ax.set_xticks([]); ax.set_yticks([])
        if ri==0: ax.set_title(t,fontsize=11,fontweight='bold')
fig.legend(handles=[mpatches.Patch(color=(230/255,126/255,34/255),label='Vaskular'),
                    mpatches.Patch(color=(46/255,204/255,113/255),label='Embrio')],
           loc='lower center',ncol=2,fontsize=10)
plt.tight_layout(rect=[0,0.05,1,0.95]); plt.savefig('/content/out/fig_unet_compare.png',dpi=150,bbox_inches='tight'); plt.show()


## 5. Simpan ke Drive

In [ ]:
import shutil
OUT='/content/drive/MyDrive/duck_egg_unet_retrained'; os.makedirs(OUT,exist_ok=True)
shutil.copy('/content/out/model.pth',OUT)
shutil.copy('/content/out/fig_unet_compare.png',OUT)
print('Tersimpan ke',OUT,'- model.pth + fig_unet_compare.png')
print('\nUnduh model.pth, taruh di laptop sebagai models/unet_retrained/model.pth,')
print('lalu notebook evaluasi & figure pakai checkpoint ini.')


---
**Catatan jujur:** embrio (blob) biasanya bisa Dice 0.4-0.7; vaskular (tipis, label heuristik)
sering tetap rendah. Laporkan angka Test Dice yang **benar-benar keluar** di sel 4, jangan 0.874.